In [1]:
import json

import colorcet
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import scipy.stats

from lib.plots import set_axis, attach_colorbar, kde1d_at
from lib.stats import fit_power_law

In [3]:
task_id = "task-4"

metrics_table = pl.read_csv(f"{task_id}/metrics.csv")

with open(f"{task_id}/config_template.json") as file:
    config_template = json.load(file)

In [4]:
sweep_ranges = config_template["@meta"]["sweep_ranges"]

min_valency, max_valency = sweep_ranges["association_valency:int"]

key_label = {
    "acetylation_level":    "Ac. level, " + r"$ \phi $",
    "association_lifetime": "Lifetime,\n" + r"$ 1 / k_\mathrm{dissoc} $",
    "msd_alpha":            "MSD exp., " + r"$ \alpha $",
    "msd_alpha_a":          "Ac. MSD exp., " + r"$ \alpha_\mathrm{A} $",
    "msd_alpha_b":          "Non-ac. MSD\nexp., " + r"$ \alpha_\mathrm{B} $",
}

KeyError: 'association_valency:int'

In [ ]:
fig, ax = plt.subplots(figsize=(2.1, 1.5))

valency_values = np.arange(min_valency, max_valency + 1)
min_alpha, max_alpha = 0.2, 0.7

y_key = "msd_alpha"
z_key = "acetylation_level"

sm = plt.cm.ScalarMappable(
    norm=plt.cm.colors.LogNorm(*sweep_ranges[z_key + ":logreal"]),
    cmap=colorcet.m_rainbow,
)
random = np.random.Generator(np.random.PCG64(1))

def scale_density(density: np.ndarray) -> np.ndarray:
    return 0.05 * np.sqrt(density)

for v in valency_values:
    section = (
        metrics_table
        .filter(pl.col("association_valency") == v)
        .filter(pl.col(y_key).is_finite())
    )

    x = np.full(len(section), v, dtype=np.float32)
    y = section[y_key]
    z = section[z_key]

    kde = scipy.stats.gaussian_kde(y, bw_method=0.15)

    widths = scale_density(kde(y))
    x += random.uniform(-widths, widths, size=x.shape)

    env_dx = 0.9 * scale_density(kde(y))
    x_env_min = v - env_dx
    x_env_max = v + env_dx
    x = np.clip(x, x_env_min, x_env_max)

    ax.scatter(x, y, c=z, s=1, norm=sm.norm, cmap=sm.cmap, zorder=3)

    y = np.linspace(min_alpha, max_alpha, num=50)
    dx = scale_density(kde(y))
    x_min = v - dx
    x_max = v + dx
    ax.fill_betweenx(y, x_min, x_max, lw=0.3, ec="#222", fc="w", zorder=2)


ax.axhline(0.56, ls="-", lw=0.5, color="C:y", zorder=1, label="fast")
ax.axhline(0.40, ls="-", lw=0.5, color="C:g", zorder=1, label="slow")

ax.grid(True, zorder=1)
ax.set_xticks(valency_values)
ax.set_xlim(min_valency - 1, max_valency + 1)
ax.set_ylim(min_alpha, max_alpha)

cbar = attach_colorbar(ax, sm)
cbar.set_label(key_label.get(z_key, z_key), usetex=True)

ax.set_xlabel(r"Valency, $ z $", usetex=True)
ax.set_ylabel(key_label[y_key], usetex=True)
ax.set_title(task_id, fontsize="small")

pass